# Project: PageRank of my vault 🕸️
Your Obsidian vault **is** a graph: notes are nodes, `[[wikilinks]]` are edges. Which notes are the hubs of your knowledge? Which are orphans nobody links to? Implement link parsing and **PageRank** (Google's original algorithm), check it against NetworkX, and get link suggestions for orphans.

Topic: [[Graph ML Overview]], [[Link Prediction]] · guide note: [[Project - PageRank of My Vault]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import pathlib, re, numpy as np, pandas as pd, matplotlib.pyplot as plt
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

VAULT = pathlib.Path("../..").resolve()
FOLDERS = ["Machine Learning", "9-semester/IT-lov"]
files = {f.stem: f for d in FOLDERS for f in (VAULT / d).rglob("*.md") if "Flashcards" not in f.parts}
print(len(files), "notes")

## 1. Parse wikilinks
Return the **set** of link targets in a note's text: `[[Note]]`, `[[Note|alias]]`, `[[Note#Heading]]` and `[[folder/Note]]` all point to `Note`.

In [ ]:
def wikilinks(text):
    # TODO 1: regex for [[...]], strip |alias and #heading, keep the last path component
    raise NotImplementedError  # TODO 1

In [ ]:
check("wikilinks", lambda: wikilinks("See [[PCA]], [[PCA|principal components]], [[CNN#Pooling]] and [[8-semester/ML/cheetsheet|sheet]].") == {"PCA", "CNN", "cheetsheet"},
      "Expect {'PCA', 'CNN', 'cheetsheet'}.")
if ready("wikilinks"):
    names = sorted(files); ix = {n: i for i, n in enumerate(names)}
    A = np.zeros((len(names), len(names)))
    for n, f in files.items():
        for t in wikilinks(f.read_text(encoding="utf-8")):
            if t in ix and t != n: A[ix[n], ix[t]] = 1
    print(f"{int(A.sum())} links between {len(names)} notes; {int((A.sum(0) == 0).sum())} notes have no incoming links")

## 2. PageRank
A random surfer follows a random outgoing link with probability $d=0.85$, or teleports to a random note. Notes with **no** outgoing links ("dangling") teleport uniformly. Iterate
$$r \leftarrow \frac{1-d}{n} + d\Big(\sum_{j\to i}\frac{r_j}{\text{out}_j} + \frac{1}{n}\sum_{j\ \text{dangling}} r_j\Big)$$
`A[i, j] = 1` means note *i* links to note *j*.

In [ ]:
def pagerank(A, d=0.85, iters=100):
    # TODO 2: power iteration with teleportation and dangling-node handling; return a vector that sums to 1
    raise NotImplementedError  # TODO 2

In [ ]:
import networkx as nx
def _t_pr():
    A = np.array([[0, 1, 1, 0], [0, 0, 1, 0], [1, 0, 0, 0], [0, 0, 1, 0.0]]); A[3] = 0      # node 3 dangling
    ref = nx.pagerank(nx.from_numpy_array(A, create_using=nx.DiGraph), alpha=0.85, tol=1e-12)
    r = pagerank(A); return np.isclose(r.sum(), 1) and np.allclose(r, [ref[i] for i in range(4)], atol=1e-6)
check("pagerank", _t_pr, "Must sum to 1 and match networkx.pagerank (which also spreads dangling mass uniformly).")
if ready("wikilinks", "pagerank"):
    r = pagerank(A)
    top = pd.DataFrame({"note": names, "PageRank": r, "in-links": A.sum(0).astype(int)}).sort_values("PageRank", ascending=False)
    print(top.head(15).to_string(index=False))
    print("\nOrphans (no incoming links):", [n for n in names if A[:, ix[n]].sum() == 0][:15])

## 3. Suggest links for orphans (Adamic–Adar)
Two notes that link to the same *rare* notes probably belong together: score $\sum_{w\in N(u)\cap N(v)} 1/\log|N(w)|$ (undirected neighbours).

In [ ]:
if ready("wikilinks", "pagerank"):
    G = nx.from_numpy_array(np.maximum(A, A.T)); G = nx.relabel_nodes(G, dict(enumerate(names)))
    orphans = [n for n in names if A[:, ix[n]].sum() == 0 and G.degree(n) > 0][:6]
    for o in orphans:
        cand = sorted(((p, s) for _, p, s in nx.adamic_adar_index(G, [(o, v) for v in names if v != o and not G.has_edge(o, v)])), key=lambda x: -x[1])[:3]
        print(f"{o:>45} → link it from: {[c for c, _ in cand]}")

<details><summary>▶ Solution</summary>

```python
def wikilinks(text):
    links = set()
    for m in re.findall(r"\[\[([^\]]+)\]\]", text):
        target = m.split("|")[0].split("#")[0].strip()
        target = target.split("/")[-1]
        if target:
            links.add(target)
    return links
```

```python
def pagerank(A, d=0.85, iters=100):
    n = A.shape[0]
    out = A.sum(1)
    r = np.full(n, 1 / n)
    for _ in range(iters):
        spread = np.where(out > 0, r / np.maximum(out, 1), 0) @ A
        dangling = r[out == 0].sum() / n
        r = (1 - d) / n + d * (spread + dangling)
    return r
```
</details>

## Stretch goals
- Run it on the **whole** vault (all semesters). Which course is most connected?
- Personalised PageRank: teleport only to the note you're studying, giving "notes to read next".
- Plot the graph with node size ∝ PageRank (`nx.draw_spring`), coloured by folder.